In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from pixelator import read_pna as read
from pixelator.pna.plot import molecule_rank_plot

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_style("whitegrid")
import pandas as pd
from pixelator.common.statistics import clr_transformation, dsb_normalize
import scanpy as sc
from pixelator.mpx.plot import density_scatter_plot
import scanpy.external as sce
import networkx as nx

from sklearn.decomposition import PCA
from sklearn.preprocessing import QuantileTransformer
from sklearn.neighbors import NearestNeighbors
from scipy.stats import norm
from pixelator.common.statistics import clr_transformation, dsb_normalize

from pixelator.pna.analysis import calculate_differential_proximity
from rich import print
import anndata as ad
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection
from statsmodels.stats.multitest import multipletests

import sys
sys.path.append('/home/projects/nyosef/zvise/PixelGen/PixelGen')

from multimodalvi import MultiModalSCVI
from multimodalvae import MultiModalVAE, AggMethod, D
from enums import AggMethod, D
from metrics import MultiModalVIMetrics
from sklearn.preprocessing import PowerTransformer
from pxl_utils import train_model, get_model_latents, convert_polarization_to_feature_matrix, \
     convert_colocalization_to_feature_matrix, download_pxl
from scvi_utils import plot_losses, pca_neighbors_umap, calc_PCA, add_one_hot_encoding_obsm, plot_cumulative_variance
from common_utils import standardize, std_clip, filter_hv, split_pair_column, filter_df_by_two_columns, rank_plot
from metrics import MultiModalVIMetrics, distr_autocorrelation_in_latent
from multimodalvi import MultiModalSCVI
from multimodalvae import MultiModalVAE, AggMethod, D
from enums import AggMethod, D
from scvi.model import SCVI


In [ ]:
DATA_DIR=Path('/home/projects/nyosef/zvise/PixelGen/PixelGen/Data/')
files = [f for f in DATA_DIR.rglob('*.pxl') if f.is_file()]
data = read(files)
adata=data.adata()
adata.obs['condition'] = adata.obs['sample'].apply(lambda x: 'PHA' if 'PHA' in x else ('unstim' if 'unstim' in x else None))
adata = adata[adata.obs["tau_type"] == "normal"].copy()


In [ ]:
adata.obs.condition.value_counts()

# Data audit and preprocess

In [ ]:
molecule_rank_df = adata.obs[["condition", "n_umi"]].copy()
molecule_rank_df["rank"] = molecule_rank_df.groupby(["condition"])["n_umi"].rank(
    ascending=False, method="first"
)
fig_intersection, ax = molecule_rank_plot(molecule_rank_df, group_by="condition")
molecule_thresh = {
    'PHA': 45000,
    'unstim': 25000,
   
}
for sample, thresh in molecule_thresh.items():
    ax.axhline(thresh, color='black', linestyle='--')

In [ ]:
mask_by_rank=False
if mask_by_rank:
    mask = np.zeros(adata.n_obs, dtype=bool)

    for cond, thresh in molecule_thresh.items():
        cond_mask = adata.obs['condition'] == cond
        umi_mask = adata.obs['n_umi'] >= thresh
        mask |= cond_mask & umi_mask  # keep if both condition and umi threshold satisfied

    # Apply filter
    adata_filtered = adata[mask].copy()

    print(f"Kept {adata_filtered.n_obs:,} of {adata.n_obs:,} components after filtering.")
else:
    print (print(f"Kept {adata.n_obs:,} of {adata.n_obs:,} components after filtering."))

In [ ]:
# whitelist = []   
# detection_thresh = 0.05                


# X = adata.X.A if hasattr(adata.X, "A") else (
#     adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X
# )

# # --- detection rate per protein ---
# det_rate = (X > 0).sum(axis=0) / X.shape[0]

# # --- mark proteins ---
# adata.var['detection_rate'] = det_rate
# adata.var['whitelisted'] = adata.var_names.isin(whitelist)
# adata.var['keep_protein'] = (adata.var['detection_rate'] >= detection_thresh) | adata.var['whitelisted']

# # --- list of deleted markers ---
# deleted_markers = adata.var_names[~adata.var['keep_protein']].tolist()
# print(f"Deleted {len(deleted_markers)} markers:")
# print(deleted_markers)

# # --- filter proteins ---
# adata = adata[:, adata.var['keep_protein']].copy()


In [ ]:
df = pd.DataFrame(adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X,
                  index=adata.obs_names,
                  columns=adata.var_names)

# Compute global summary statistics
summary = df.describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]]
summary.head()

In [ ]:
adata.obs["total_counts"] = np.sum(adata.X, axis=1).A1 if hasattr(adata.X, "A1") else np.sum(adata.X, axis=1)

# summary statistics
summary = adata.obs["total_counts"].describe()
print(summary)

In [ ]:
adata.layers['counts']=adata.X
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers['norm_log1p']=adata.X


In [ ]:
adata.layers['log1p']=adata.obsm['log1p']

# Annotation

In [ ]:
working_layer='norm_log1p'


sc.pp.pca(adata, n_comps=30, use_highly_variable=False, layer=working_layer)
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=20, use_rep=None)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.6)

sc.pl.umap(adata, color=["condition",'leiden'], legend_loc='on data',frameon=False, size=30, show=False)
plt.gcf().set_size_inches(18, 10)
plt.title("UMAP of Leiden clusters in MRVI U-space", fontsize=16, pad=15)
plt.show()

In [ ]:
adata_tmp = ad.AnnData(
    X=adata.layers[working_layer],
    obs=adata.obs.copy(), var=adata.var.copy()
)
sc.tl.rank_genes_groups(adata_tmp,'leiden', method="wilcoxon",)


diff_exp_df = sc.get.rank_genes_groups_df(adata_tmp, group=None)
diff_exp_df["-log10(adjusted p-value)"] = -np.log10(diff_exp_df["pvals_adj"])
diff_exp_df["Significant"] = diff_exp_df["pvals_adj"] < 0.01
diff_exp_df.head()

In [ ]:
df = diff_exp_df.pivot(index=["names"], columns=["group"], values=["logfoldchanges"])

markers_for_heatmap = set(
    diff_exp_df[
        (np.abs(diff_exp_df["logfoldchanges"]) > 5) & diff_exp_df["Significant"]
    ]["names"]
)
markers_to_add=[]

markers_for_heatmap.update(markers_to_add)

df = df[df.index.isin(markers_for_heatmap)]

df.columns = [cluster for _, cluster in df.columns]
fig = sns.clustermap(df, yticklabels=True, linewidths=0.1, cmap="vlag", vmin=-5, vmax=5);
fig.fig.set_size_inches(10, 15)

In [ ]:
cell_annotations = {
    "0": "CD4_unstim",
    "4": "NK_unsim",
    "1": "CD4_PHA",
    "2": "CD8_PHA",
    "3": "platelet-monocyte_umstim",
    "4": "NK_unstim",
    "5": "CD4_PHA",
    "6": "CD8_ustim",
    "7": "B_naive_PHA",
    "8": "Platelets",
    "9": "APC_active_PHA",
    "10": "CD4_unstim",
    "11": "CD8_PHA",
    "12": "CD4_PHA",
    "13": "CD4_unstim",
    "14": "NK_PHA",
    "15": "B_naive_unstim",
    "16": "basophil–monocyte_unstim",
    "17": "Basophiles",

     
}


adata.obs["cell_type"] = adata.obs[
    "leiden"
].map(cell_annotations)

In [ ]:
sc.pl.umap(adata, color=["condition",'cell_type'], legend_loc='on data',frameon=False, size=40,ncols=1,show=False)
plt.gcf().set_size_inches(20, 12)
plt.title("Cell type clustering", fontsize=16, pad=15)
plt.show()

# SPATIAL

In [ ]:
spatial=data.proximity().to_df()
print (spatial.shape)
spatial = spatial[
    (spatial["min_count"]>=50) &
    (spatial["join_count_expected_mean"]>=10)
]
spatial = spatial[spatial["component"].isin(adata.obs.index)].copy()

print (spatial.shape)

In [ ]:
# PROTEIN_PREVALENCE_FRAC = 0.05   

# n_components = adata.n_obs
# min_components = max(1, int(np.ceil(PROTEIN_PREVALENCE_FRAC * n_components)))

# prot_counts_1 = spatial.reset_index().groupby('marker_1')['component'].nunique()
# prot_counts_2 = spatial.reset_index().groupby('marker_2')['component'].nunique()

# prot_counts = prot_counts_1.add(prot_counts_2, fill_value=0).astype(int)

# proteins_to_keep = prot_counts[prot_counts >= min_components].index.tolist()
# proteins_to_drop = prot_counts[prot_counts < min_components].index.tolist()

# print(f"[Proteins] kept: {len(proteins_to_keep)}, dropped: {len(proteins_to_drop)}")
# print("Dropped proteins:", proteins_to_drop)

# uncut_size=spatial.shape[0]
# print (spatial.shape)

# spatial = spatial[
#     spatial['marker_1'].isin(proteins_to_keep) &
#     spatial['marker_2'].isin(proteins_to_keep)
# ].copy()

# cut_size=spatial.shape[0]
# print (spatial.shape)
# print (f'propotion of remaining pairs:{cut_size / uncut_size}')

In [ ]:
spatial[(spatial.marker_1=='B2M')&(spatial.marker_2=='CD11a')&(spatial.component=='9567c4b0b99c07ab')]


In [ ]:
def _pivot_joinz(spatial, col="join_count_z"):
    # pair label "marker_1/marker_2" (already ordered as user promised)
    tmp = spatial.reset_index().copy()  # expects index name 'component'
    tmp["pair"] = tmp["marker_1"].astype(str) + "/" + tmp["marker_2"].astype(str)
    wide = tmp.pivot_table(index="component", columns="pair", values=col, aggfunc="first")
    return wide

def build_spz_obsm(
    adata,
    spatial,                 
    col="join_count_z",
    asinh_scale=3.0
):
    wide = _pivot_joinz(spatial, col=col)
    wide = wide.reindex(index=adata.obs_names)  # align with adata.obs
    
    wide = wide.fillna(0.0)

    adata.obsm["spatial_id"] = wide.astype(np.float32)

    a = float(asinh_scale)
    adata.obsm["spatial_arcsinh"] = np.arcsinh(wide / a).astype(np.float32)

    print(f"Stored DataFrames in .obsm with shape {wide.shape} (cells × pairs)")


build_spz_obsm(
    adata,
    spatial,            # long-form table (index='component')
    col="join_count_z",
    asinh_scale=3.0
)


In [ ]:
spatial_df = adata.obsm["spatial_arcsinh"]

tmp = ad.AnnData(X=spatial_df.values, obs=adata.obs.copy(), var=pd.DataFrame(index=spatial_df.columns))

sc.pp.highly_variable_genes(tmp, n_top_genes=2000, flavor="seurat", subset=False)

top_cols = tmp.var[tmp.var["highly_variable"]].index

adata.obsm["spatial_arcsinh_2000"] = spatial_df[top_cols].copy()

print(f"Selected {len(top_cols)} most variable spatial-pair features using Scanpy HVG.")


In [ ]:

sc.pp.neighbors(adata, n_neighbors=15, use_rep="spatial_arcsinh_2000")
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.4, random_state=0, key_added='leiden_spatial')
sc.pl.umap(
    adata,
    color=['condition','cell_type','leiden_spatial'],
    ncols=1,
    legend_loc='on data',
    show=False,
    
)
plt.gcf().set_size_inches(14, 16)

plt.tight_layout()
plt.show()

In [ ]:
ADATA_FOR_MODEL_PATH='/home/projects/nyosef/zvise/PixelGen/PixelGen/Data/adatas/PBMC_adata_for_model.h5ad'
adata.write_h5ad(ADATA_FOR_MODEL_PATH)

# MODEL

In [ ]:
ADATA_FOR_MODEL_PATH='/home/projects/nyosef/zvise/PixelGen/PixelGen/Data/adatas/PBMC_adata_for_model.h5ad'

load_model=True
if load_model:
    adata=sc.read_h5ad(ADATA_FOR_MODEL_PATH)
    adata
else:
    print(adata)
    
adata.obsm['pca'] = PCA(n_components=30).fit_transform(adata.X)


### REGULAR MODEL

In [ ]:
model_cls = MultiModalSCVI
abundance_layer = 'norm_log1p'
spatial_layer='spatial_id'
max_epochs=10000

latent_name=f'weighted_latent'
    
setup_kwargs = dict(layer=abundance_layer, extra_modality_keys=[spatial_layer], n_modalities=2, batch_key='condition', spatial_mask_key=None,  )
model_kwargs = dict(n_latent=30, n_hidden=128, n_layers=2, dropout_rate=0.1, 
                            distrs=[D.Normal, D.Normal], 
                            
                            loss_weights='auto',
                            joint_kl=False, unimodal_kl=True,
                            
                            decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp')
                        )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                        early_stopping_patience=200, batch_size=2000,
                        max_epochs=max_epochs, enable_checkpointing=True, 
                        plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400)
                    )
model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)

In [ ]:
model.save('/home/projects/nyosef/zvise/PixelGen/PixelGen/Data/models/all_spatial_id',overwrite=True)

In [ ]:
imputed = model.get_normalized_expression(
    adata=adata,
    return_mean_expression=True,
    return_l2_error=True,
    return_px_distrs=False,
    return_numpy=True
)

In [ ]:
adata_tmp = ad.AnnData(
    X=adata.layers['norm_log1p'],
    obs=adata.obs.copy(), var=adata.var.copy()
)
sc.tl.rank_genes_groups(adata_tmp,'leiden_z', method="wilcoxon",)


diff_exp_df = sc.get.rank_genes_groups_df(adata_tmp, group=None)
diff_exp_df["-log10(adjusted p-value)"] = -np.log10(diff_exp_df["pvals_adj"])
diff_exp_df["Significant"] = diff_exp_df["pvals_adj"] < 0.01
df = diff_exp_df.pivot(index=["names"], columns=["group"], values=["logfoldchanges"])

markers_for_heatmap = set(
    diff_exp_df[
        (np.abs(diff_exp_df["logfoldchanges"]) > 5) & diff_exp_df["Significant"]
    ]["names"]
)
markers_to_add=[]

markers_for_heatmap.update(markers_to_add)

df = df[df.index.isin(markers_for_heatmap)]

df.columns = [cluster for _, cluster in df.columns]
fig = sns.clustermap(df, yticklabels=True, linewidths=0.1, cmap="vlag", vmin=-5, vmax=5);
fig.fig.set_size_inches(10, 15)

In [ ]:
metrics=MultiModalVIMetrics(
    adata,
    {'model':model},
    pca_key='pca',
    batch_key='condition'
)

metrics.run()

In [ ]:
_ = metrics.mean_modality_errors_barplot(reconstruction_mean=True)

In [ ]:
metrics.plot_negative_likelihood()

In [ ]:
df = metrics.plot_latent_quality_metrics()
display (df)

In [ ]:
metrics.plot_scib_metrics()

### BATCH EFFECT MODEL

In [ ]:
model_cls = MultiModalSCVI
abundance_layer = 'norm_log1p'
spatial_layer='spatial_id'
max_epochs=10000

latent_name=f'weighted_latent'
    
setup_kwargs = dict(layer=abundance_layer, extra_modality_keys=[spatial_layer], n_modalities=2, batch_key='condition', spatial_mask_key=None,  )
model_kwargs = dict(n_latent=30, n_hidden=128, n_layers=2, dropout_rate=0.1, 
                            distrs=[D.Normal, D.Normal], 
                            
                            loss_weights='auto',
                            joint_kl=False, unimodal_kl=True,
                            
                            decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp')
                        )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                        early_stopping_patience=200, batch_size=2000,
                        max_epochs=max_epochs, enable_checkpointing=True, 
                        plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400)
                    )
model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)

In [ ]:
z=model.get_latent_representation(adata,)

adata.obsm['z']=z

sc.pp.neighbors(adata, n_neighbors=15, use_rep="z")
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.8, random_state=0, key_added='leiden_z')
sc.pl.umap(
    adata,
    color=['condition','cell_type','leiden_z'],
    ncols=1,
    show=False,
    
)
plt.gcf().set_size_inches(14, 16)

plt.tight_layout()
plt.show()

### SCVI MODEL

In [ ]:
adata

In [ ]:
SCVI.setup_anndata(adata, layer='counts', batch_key="condition")
model = SCVI(adata, n_latent=30)
model.train()

In [ ]:
latent = model.get_latent_representation()
adata.obsm["X_scVI"] = latent

In [ ]:
sc.pp.neighbors(adata, use_rep="X_scVI")
sc.tl.umap(adata)
sc.pl.umap(adata, color=["condition", "cell_type"], legend_loc='on data',frameon=False, show=False)
plt.gcf().set_size_inches(18, 10)
plt.title("UMAP of Leiden clusters in MRVI U-space", fontsize=16, pad=15)
plt.show()

# SPATIAL

In [ ]:
adata.obs.cell_type.value_counts()

In [ ]:
adata.obsm['spatial_id']

In [ ]:
cell_types

In [ ]:
spatial_df = pd.DataFrame(adata.obsm["spatial_id"], index=adata.obs_names)
cell_types = adata.obs["cell_type"]

group1 = spatial_df[cell_types == "CD4_unstim"]
group2 = spatial_df[cell_types == "CD4_PHA"]

results = []
for col in spatial_df.columns:
    vals1 = group1[col].dropna()
    vals2 = group2[col].dropna()
    if len(vals1) > 2 and len(vals2) > 2:  # avoid empty groups
        stat, pval = stats.ttest_ind(vals1, vals2, equal_var=False)
        diff = np.mean(vals2) - np.mean(vals1)
        results.append((col, diff, pval))

results_df = pd.DataFrame(results, columns=["feature", "mean_diff", "pval"])
results_df["padj"] = multipletests(results_df["pval"], method="fdr_bh")[1]
results_df["log10p"] = -np.log10(results_df["pval"])
results_df["abs_diff"] = results_df["mean_diff"].abs()

plt.figure(figsize=(8,6))
plt.scatter(results_df["mean_diff"], results_df["log10p"],
            c=(results_df["padj"] < 0.05), cmap="coolwarm", alpha=0.8)
plt.axhline(-np.log10(0.05), color="grey", ls="--", lw=1)
plt.xlabel("Mean difference (cd4_PHA − cd4_unstim)")
plt.ylabel("−log10(p-value)")
plt.title("Spatial feature differences between CD4_unstim vs CD4_PHA")

top_features = results_df.sort_values("abs_diff", ascending=False).head(5)
for _, row in top_features.iterrows():
    plt.text(row["mean_diff"], row["log10p"], row["feature"], fontsize=8)

plt.tight_layout()
plt.show()



In [ ]:
results_df.sort_values(["pval", "abs_diff"], ascending=[True, False]).head(10)


In [ ]:
# look at abundance
# take top 10 for activated\unactuvated
# size of dot should be abundance